<a href="https://colab.research.google.com/github/TageYassir/Big-Data-RAG-/blob/main/Big_data_courses_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
%pip install -q sentence-transformers gradio pypdf
print("✅ Installation terminée !")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 11.7 MB/s eta 0:00:00
✅ Installation terminée !


In [4]:
%pip install gdown pypdf torch pdfplumber

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 62.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 55.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 43.8 MB/s eta 0:00:00
  Attempting uninstall: Pillow
    Found existing installation: pillow 11.3.0
    Uninstalling pillow-11.3.0:
      Successfully uninstalled pillow-11.3.0


In [5]:
!apt-get install -y tesseract-ocr tesseract-ocr-fra poppler-utils
!pip install -q pytesseract pdf2image pypdf

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (5.3.4-1build5).
The following NEW packages will be installed:
  poppler-utils tesseract-ocr-fra
0 upgraded, 2 newly installed, 0 to remove and 25 not upgraded.
Need to get 796 kB of archives.
After this operation, 1,880 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 poppler-utils amd64 24.02.0-1ubuntu9.9 [212 kB]
Get:2 http://archive.ubuntu.com/ubuntu noble/universe amd64 tesseract-ocr-fra all 1:4.1.0-2 [584 kB]
Fetched 796 kB in 2s (451 kB/s)
Selecting previously unselected package poppler-utils.
(Reading database ... 129182 files and directories currently installed.)
Preparing to unpack .../poppler-utils_24.02.0-1ubuntu9.9_amd64.deb ...
Unpacking poppler-utils (24.02.0-1ubuntu9.9) ...
Selecting previously unselected package tesseract-ocr-fra.
Preparing to unpack .../tesseract-ocr-fra_1%

In [12]:
import torch
from transformers import pipeline

MODELE = "Qwen/Qwen2.5-1.5B-Instruct" # Ou "Qwen/Qwen2.5-1.5B-Instruct"
DEVICE = "cuda"

generateur = pipeline(
    "text-generation",
    model=MODELE,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("🚀 Modèle rapide chargé avec succès !")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

🚀 Modèle rapide chargé avec succès !


In [60]:
import torch
import gc

# Libération de la mémoire
gc.collect()
torch.cuda.empty_cache()

print("✅ Cache GPU nettoyé !")

✅ Cache GPU nettoyé !


In [6]:
import os
import pytesseract
from google.colab import drive
from pdf2image import convert_from_path
from pypdf import PdfReader

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Path to your folder inside Google Drive
DRIVE_FOLDER_PATH = "/content/drive/MyDrive/Eidia 4/Méga-données/test"

TEXTES = {}

if os.path.exists(DRIVE_FOLDER_PATH):
    for root, _, files in os.walk(DRIVE_FOLDER_PATH):
        for file in sorted(files):
            if file.lower().endswith(".pdf"):
                filepath = os.path.join(root, file)
                extracted_text = ""

                # Step A: Attempt standard text extraction via pypdf
                try:
                    reader = PdfReader(filepath)
                    pages = reader.pages
                    num_pages = len(pages)
                    extracted_text = "\n".join((p.extract_text() or "") for p in pages).strip()
                except Exception as e:
                    num_pages = 0
                    print(f"⚠️ pypdf failed for {file}: {e}")

                # Step B: Fallback to OCR if pypdf extracted very little/no text (< 200 chars)
                if len(extracted_text) < 200:
                    try:
                        images = convert_from_path(filepath)
                        num_pages = len(images)
                        ocr_pages = [
                            pytesseract.image_to_string(img, lang="fra+eng")
                            for img in images
                        ]
                        extracted_text = "\n".join(ocr_pages).strip()
                        tag = "🔍 [OCR]"
                    except Exception as e:
                        tag = "❌ [OCR Error]"
                        print(f"❌ OCR failed for {file}: {e}")
                else:
                    tag = "✅ [PDF Text]"

                TEXTES[file] = extracted_text
                print(f"{tag} {file:50s} {num_pages:3d} page(s) · {len(extracted_text):6d} caractères")

    print(f"\n📚 {len(TEXTES)} documents chargés.")
else:
    print(f"❌ Folder not found at {DRIVE_FOLDER_PATH}. Make sure the path is correct.")

Mounted at /content/drive
✅ [PDF Text] Atelier_1 (Apache Kafka Cloud).pdf                  21 page(s) ·  30883 caractères
✅ [PDF Text] Atelier_2 (Apache Kafka Local).pdf                  19 page(s) ·  39778 caractères
✅ [PDF Text] Atelier_3.0 (Apache Hadoop - Windows & Local).pdf   23 page(s) ·  43523 caractères
✅ [PDF Text] Atelier_3.1 (Apache Hadoop - UNIX - Distributed).pdf  29 page(s) ·  48449 caractères
🔍 [OCR] Cours_MegaData_S1.pdf                               72 page(s) ·  40123 caractères
🔍 [OCR] Cours_MegaData_S2.pdf                               86 page(s) ·  43141 caractères
🔍 [OCR] Cours_MegaData_S3.pdf                               56 page(s) ·  30161 caractères
🔍 [OCR] Cours_MegaData_S4.pdf                               61 page(s) ·  32508 caractères
🔍 [OCR] Cours_MegaData_S5.pdf                               59 page(s) ·  35776 caractères
🔍 [OCR] Cours_MegaData_S6.pdf                               47 page(s) ·  33912 caractères
🔍 [OCR] Cours_MegaData_S7.pdf             

In [7]:
def decouper(texte, taille=1000, chevauchement=80):
    """Découpe un texte en passages d'environ `taille` caractères, en coupant de préférence en fin de phrase."""
    texte = " ".join(texte.split())          # nettoyage des espaces et sauts de ligne
    passages, debut = [], 0
    while debut < len(texte):
        fin = min(debut + taille, len(texte))
        if fin < len(texte):
            coupe = texte.rfind(". ", debut + taille // 2, fin)   # fin de phrase la plus proche
            if coupe != -1:
                fin = coupe + 1
        passages.append(texte[debut:fin].strip())
        if fin >= len(texte):
            break
        debut = max(fin - chevauchement, debut + 1)   # chevauchement
    return [p for p in passages if p]


DOCUMENTS = []
for f, t in TEXTES.items():
    # Formatage générique propre du nom de document
    nom = f.replace(".pdf", "").replace("_", " ").strip().title()

    # On ne découpe que les documents contenant du texte
    for i, p in enumerate(decouper(t), 1):
        DOCUMENTS.append({"titre": f"{nom} · passage {i}", "texte": p})

print(f"✅ {len(DOCUMENTS)} passages, prêts à être encodés.")
print()
if DOCUMENTS:
    print("Exemple ·", DOCUMENTS[0]["titre"])
    print(DOCUMENTS[0]["texte"][:350], "...")
else:
    print("⚠️ Aucun texte n'a été extrait des documents.")

✅ 553 passages, prêts à être encodés.

Exemple · Atelier 1 (Apache Kafka Cloud) · passage 1
Atelier : N°1 Apache Kafka : Cloud Configuration & Manipulation Pr. MOUNTASSER IMADEDDINE I. Création & Configuration d’un Serveur Kafka « Solution Cloud » ................................................ 2 II. Développer un Projet Kafka ........................................................................................................... 4 II ...


In [39]:
DOCUMENTS[-1]

{'titre': 'Tp1 Tagemouati Yassir · passage 47',
 'texte': 'rn "null"; if (s.length() <= 4) return "****"; return "****" + s.substring(Math. max (0, s.length() - 4)); } private static String showInvisible(String s) { if (s == null) return "null"; StringBuilder sb = new StringBuilder(); for (int i = 0; i < s.length(); i++) { char c = s.charAt(i); if (c <= \' \' || Character. getType (c) == Character. FORMAT ) { sb.append(String. format ("\\\\u%04x", (int) c)); } else { sb.append(c); } } return sb.toString(); } }'}

In [8]:
from sentence_transformers import SentenceTransformer
import numpy as np

# TROU 1 · le modèle de mardi
encodeur = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

# On encode le TITRE avec le passage : le moteur saura ainsi de quel club on parle.
# Mais le texte qu'on donnera au modèle, lui, restera propre.
textes = [f"{d['titre']}\n{d['texte']}" for d in DOCUMENTS]
vecteurs = encodeur.encode(textes, normalize_embeddings=True)

print(f"✅ {len(vecteurs)} passages encodés.")
print(f"Chaque passage est devenu un vecteur de {vecteurs.shape[1]} nombres.")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ 553 passages encodés.
Chaque passage est devenu un vecteur de 384 nombres.


In [9]:
# TROU 2 · combien de passages on donne au modèle ?
def chercher(question, k=30):
    """Renvoie les k passages les plus proches de la question."""
    v_question = encodeur.encode(question, normalize_embeddings=True)
    similarites = vecteurs @ v_question
    indices = np.argsort(-similarites)[:k]
    return [(DOCUMENTS[i]["titre"], DOCUMENTS[i]["texte"], float(similarites[i]))
            for i in indices]

for titre, texte, score in chercher("C'est quoi Kafka?"):
    print(f"[{score:.2f}] {titre}")

print()
print("✅ Le moteur retrouve les bons passages, et le bon club !")

[0.70] Atelier 2 (Apache Kafka Local) · passage 38
[0.69] Cours Megadata S2 · passage 49
[0.68] Atelier 2 (Apache Kafka Local) · passage 7
[0.67] Cours Megadata S2 · passage 31
[0.64] Cours Megadata S2 · passage 27
[0.64] Cours Megadata S2 · passage 29
[0.64] Cours Megadata S2 · passage 33
[0.62] Atelier 2 (Apache Kafka Local) · passage 39
[0.62] Atelier 2 (Apache Kafka Local) · passage 22
[0.62] Atelier 2 (Apache Kafka Local) · passage 35
[0.62] Atelier 2 (Apache Kafka Local) · passage 3
[0.61] Atelier 2 (Apache Kafka Local) · passage 4
[0.61] Atelier 2 (Apache Kafka Local) · passage 20
[0.60] Atelier 1 (Apache Kafka Cloud) · passage 13
[0.60] Atelier 2 (Apache Kafka Local) · passage 24
[0.59] Atelier 2 (Apache Kafka Local) · passage 21
[0.59] Atelier 1 (Apache Kafka Cloud) · passage 10
[0.59] Tp2 Tagemouati Yassir · passage 1
[0.58] Atelier 2 (Apache Kafka Local) · passage 11
[0.58] Atelier 3.0 (Apache Hadoop - Windows & Local) · passage 47
[0.57] Cours Megadata S2 · passage 46
[0.57

In [10]:
for titre, texte, score in chercher("How to install hadoop ?"):
    print(f"[{score:.2f}] {titre}")

[0.55] Atelier 3.1 (Apache Hadoop - Unix - Distributed) · passage 13
[0.53] Atelier 3.1 (Apache Hadoop - Unix - Distributed) · passage 15
[0.50] Atelier 3.0 (Apache Hadoop - Windows & Local) · passage 10
[0.50] Atelier 3.1 (Apache Hadoop - Unix - Distributed) · passage 50
[0.49] Tp3 Tagemouati Yassir · passage 1
[0.49] Tp Docker · passage 5
[0.48] Atelier 3.1 (Apache Hadoop - Unix - Distributed) · passage 1
[0.45] Atelier 3.1 (Apache Hadoop - Unix - Distributed) · passage 4
[0.45] Atelier 3.1 (Apache Hadoop - Unix - Distributed) · passage 34
[0.45] Atelier 3.0 (Apache Hadoop - Windows & Local) · passage 5
[0.44] Atelier 3.1 (Apache Hadoop - Unix - Distributed) · passage 6
[0.43] Atelier 3.1 (Apache Hadoop - Unix - Distributed) · passage 26
[0.42] Atelier 3.1 (Apache Hadoop - Unix - Distributed) · passage 35
[0.42] Atelier 3.1 (Apache Hadoop - Unix - Distributed) · passage 3
[0.41] Atelier 3.1 (Apache Hadoop - Unix - Distributed) · passage 8
[0.41] Atelier 3.1 (Apache Hadoop - Unix - Di

In [13]:
from transformers import pipeline
import transformers

transformers.logging.set_verbosity_error()   # on masque les avertissements techniques

print("Chargement du modèle (1 à 3 minutes la première fois)...")
generateur = pipeline("text-generation", model=MODELE, device=DEVICE)

# On règle la génération une fois pour toutes
generateur.tokenizer.clean_up_tokenization_spaces = False
generateur.model.generation_config.max_new_tokens = 300
generateur.model.generation_config.do_sample = False
generateur.model.generation_config.temperature = None
generateur.model.generation_config.top_p = None
generateur.model.generation_config.top_k = None

print("✅ Modèle chargé !")



Chargement du modèle (1 à 3 minutes la première fois)...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

✅ Modèle chargé !


In [14]:
def demander_au_modele(messages):
    sortie = generateur(messages)
    return sortie[0]["generated_text"][-1]["content"]

In [15]:
# Dans toute vraie application, l'assistant a un RÔLE. Le nôtre :
ROLE = (
    "Tu es un expert et assistant pédagogique spécialiste en Big Data, architectures distribuées, "
    "Hadoop, YARN, MapReduce, Spark et Kafka.\n"
    "Tu maîtrises parfaitement le programme et les supports de cours de M. Imadeddine Mountasser.\n\n"
    "CONSIGNES DE RÉPONSE :\n"
    "1. Base tes réponses uniquement et rigoureusement sur les extraits de documents fournis dans le contexte.\n"
    "2. Sois précis, technique, concis et direct. Réponds avec rigueur académique et professionnelle.\n"
    "3. Si l'information ne se trouve pas explicitement dans les documents, indique clairement que "
    "le sujet n'est pas couvert dans les supports fournis au lieu d'inventer."
)

In [16]:
# votre question : sa réponse est dans nos guides, mais le modèle ne les a pas encore.
question_test = "what is the best version for hadoop and the steps to download it ?"

# On garde cette réponse de côté : on la comparera tout à l'heure.
reponse_sans_documents = demander_au_modele([
    {"role": "system", "content": ROLE},          # il est l'assistant du club...
    {"role": "user", "content": question_test},   # ...mais il n'a aucun document.
])
print(reponse_sans_documents)

The "best" version of Hadoop can vary depending on specific requirements such as compatibility with existing systems, ease of use, security considerations, and performance needs. However, a widely recommended version is Hadoop 3.x, which was released in 2019. This version introduced several improvements over its predecessor, including better support for modern Java versions, improved security features, and enhanced fault tolerance.

To download Hadoop 3.x, you should follow these general steps:

### Step 1: Ensure Compatibility
Before downloading, ensure that your system meets the minimum requirements specified by the latest release notes for Hadoop 3.x. These typically include:
- Java Development Kit (JDK) 8 or later
- A recent operating system like Linux, macOS, or Windows

### Step 2: Downloading the Distribution
You have two main options for obtaining the distribution:
- **Download from Apache Maven**: The official repository provides pre-built JAR files and binaries. Navigate to t

In [17]:
def repondre(question):
    """L'assistant complet : recherche + rédaction sous contrainte."""
    passages = chercher(question) # on recherches les K passages les plus pertinents vis-a-vis de la question
    contexte = "\n\n" .join(f"### {t}\n{x}" for t, x, _ in passages) # on mets tous ces "passages" au sein d'une meme string, ca sera notre "contexte"

    # On Construit le Prompt Final : qu'est-ce qu'on lui interdit ? et que doit-il dire s'il ne sait pas ?
    messages = [
        {"role": "system", "content": ROLE,
        "role": "user", "content": f"Documents :\n{contexte}\n\nQuestion : {question}"},
    ]

    reponse = demander_au_modele(messages)
    sources = ", ".join(t for t, _, _ in passages)
    return reponse, sources


# La MÊME question, cette fois avec les documents. On affiche les deux réponses l'une sous l'autre.
reponse_avec_documents, sources = repondre(question_test)

print("❓", question_test)
print()
print("❌ SANS les documents :")
print("   ", " ".join(reponse_sans_documents.split())[:400])
print()
print("✅ AVEC les documents :")
print("   ", " ".join(reponse_avec_documents.split()))
print("    📎 Sources :", sources)

❓ what is the best version for hadoop and the steps to download it ?

❌ SANS les documents :
    The "best" version of Hadoop can vary depending on specific requirements such as compatibility with existing systems, ease of use, security considerations, and performance needs. However, a widely recommended version is Hadoop 3.x, which was released in 2019. This version introduced several improvements over its predecessor, including better support for modern Java versions, improved security feat

✅ AVEC les documents :
    Based on the information provided in the document, the best version for Hadoop is: Hadoop 3.3.x The key points from the text regarding downloading Hadoop include: 1. Downloading version 3.3.6 specifically mentioned as being "stable". 2. Using the link https://dlcdn.apache.org/hadoop/common/hadoop-3.3.6/hadoop-3.3.6.tar.gz 3. Installing the downloaded tarball using `sudo tar -xvzf hadoop-3.3.6.tar.gz` So the recommended steps would be: 1. Visit the given download link. 2

In [18]:
questions = [
    "What are the different errors you can get when working on a hadoop project ?",
    "What do you need to make an hadoop map/reduce application ?",
    "What is the components of yarn and the key concept ?",
]

for q in questions:
    r, s = repondre(q)
    print(f"❓ {q}")
    print(f"💬 {r}")
    print(f"📎 {s}")
    print()

print("🎉 Levez la main dans le chat : votre assistant vient de répondre !")

❓ What are the different errors you can get when working on a hadoop project ?
💬 When working with Hadoop projects, there are several potential errors that developers may encounter. Here's a list of some common issues:

1. **Configuration Errors**:
   - Missing or incorrect settings in `core-site.xml`, `hdfs-site.xml`, or other configuration files.
   - Incorrect paths or permissions set in environment variables like `$HADOOP_HOME` or `$JAVA_HOME`.

2. **Classpath Issues**:
   - Missing required libraries or dependencies specified in `dependencies.txt`.
   - Incorrect versions of Hadoop and Java being used.

3. **Permission Denials**:
   - Insufficient permissions to execute commands or read/write files in designated directories.

4. **ZooKeeper Connection Problems**:
   - Not having ZooKeeper installed correctly or not running it properly.
   - Network connectivity issues between nodes.

5. **HDFS Path Syntax Errors**:
   - Incorrectly formatted paths in HDFS URLs or scripts.

6. **Se

In [20]:
import gradio as gr

def assistant_web(question):
    if not question.strip():
        return "Posez-moi une question sur le cours Big Data!"
    reponse, sources = repondre(question)
    return f"{reponse}\n\n📎 Sources : {sources}"

demo = gr.Interface(
    fn=assistant_web,
    inputs=gr.Textbox(label="Votre question",
                      placeholder="Ex : C'est quoi le role du zookeeper ?"),
    outputs=gr.Textbox(label="Réponse de l'assistant"),
    title="Big data RAG [Tage]",
    description="Il répond aux questions sur le cours Big Data fait par M. Imadeddine Mountasser"
)

# TROU 5 · un seul mot pour passer de votre écran au monde
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a5a53dbf16d23438d1.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
